<a href="https://colab.research.google.com/github/rene-aum/Hermes/blob/moises/Asignacion/nb3_asignacion_edas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automatización de Asignación de leads de crédito (Apis & Contingencia)
Recuerda que para este punto ya debiste subir al menos la **base de clientes con Corte 1** del día.
Además, por favor, **asegúrate de colocar correctamente**
1. La cosecha que estás asignando.
2. Si vas a escribir automáticamente en la torre de control (versión 2)

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from dateutil.relativedelta import relativedelta

id_sheets_formulario_edas = '13avOAwqLvec7h301gOQdgbA8iVKMKstH_zzggaYYFMw'
id_drive_edas = '14rh8YbJUtXiyfNdSmUWueHRyoet-SezJ'
id_drive_glomo = '1lzwrBpXAeuJLDQ2td0r2Bvmaj3iNCO_1'
id_drive_hist = '1zvW-Dxow9gz1Dnbpg_jO7my4wadDvJDW'
id_drive_catalogos = '1xQkepXoIoEHNgdYUaLAT7FtDnuzYo1qb'
id_drive_salidas = '1qZlTS_buGs876ojYKuBmOJ3n5zzWLmBS'
id_sheets_tc2 = '1k8rguLeF1O33XCaVDxPiQ1C4SbxLDSIeqNcriYtsF-k'

fh_salida =  datetime.now(ZoneInfo("America/Mexico_City")).strftime('%Y-%m-%d')   # '2026-02-17'
fh_salida_dt = datetime.strptime(fh_salida, '%Y-%m-%d')
fh_salida_dt_ma = fh_salida_dt - relativedelta(months=1)
dia_salida = str(fh_salida_dt.day).zfill(2)
mes_salida = str(fh_salida_dt.month).zfill(2)
anio_salida = str(fh_salida_dt.year).zfill(4)
mes_salida_ma = str(fh_salida_dt_ma.month).zfill(2)
anio_salida_ma = str(fh_salida_dt_ma.year).zfill(4)
fh_de_asignacion = fh_salida_dt.strftime('%d-%m-%Y')

cosecha = 'Cosecha Abr 26' #@param{type:'string'}

nb_ultimo_corte = 'ultimoCorte_edas.csv'
nb_carpeta_ctes_mes = f'{anio_salida}{mes_salida}'
nb_ctes_glomo_mm = f'BaseTotalGLOMO_{mes_salida}{anio_salida}.csv'
nb_ctes_glomo_ma = f'BaseTotalGLOMO_{mes_salida_ma}{anio_salida_ma}.csv'
nb_sheet_salida = f'Salidas {fh_salida}'

dicc_espacios = {'Reforma 510':'torre','MetrÃ³poli Patriotismo':'patriotismo','Samara SatÃ©lite':'samara', 'Gran Sur':'gran sur'}
dicc_espacios2 = {'MetrÃ³poli Patriotismo': 'Metrópoli Patriotismo','Samara SatÃ©lite': 'Samara Satélite'}
dicc_espacios3 = {'torre':'Reforma 510','patriotismo':'Metrópoli Patriotismo','samara':'Samara Satélite','gran sur':'Gran Sur'}

nivelar_carga_espacios = 0  #@param {type:"slider", min:0, max:1, step:0.05}
actualizar_tc = 'S' #@param{type:'string'}['S','N']
validar_montos = 'S' #@param{type:'string'}['S','N']

## Paqueterías

In [ ]:
import os

from_drive = True  # same flag you use everywhere

if os.environ.get("HERMES_BOOTSTRAPPED") != "1":
    # ---------- GIT ON COLAB ONLY ----------
    try:
        from google.colab import userdata

        git_token = userdata.get('gitToken')
        git_user = userdata.get('gitUser')
        git_url = f'https://{git_token}@github.com/rene-aum/Hermes.git'
        branch_to_pull = 'dev'

        os.chdir('/content')

        if not os.path.isdir('Hermes'):
            !git clone {git_url}

        %cd Hermes
        !git fetch origin {branch_to_pull}
        !git checkout {branch_to_pull}
        !git pull origin {branch_to_pull}

        !pip install -r utils/src/requirements.txt
        %cd Asignacion

    except Exception as e:
        print(e)
        print('Running in other environment not colab probably!')

    # ---------- DRIVE + SHEETS ----------
    if from_drive:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        import gspread
        from google.auth import default
        from gspread_dataframe import set_with_dataframe
        import gdown

        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)

        creds, _ = default()
        gc = gspread.authorize(creds)

    os.environ["HERMES_BOOTSTRAPPED"] = "1"
else:
    print("Bootstrap already done, assuming orchestrator ran it.")

In [ ]:
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
import sys
sys.path.append('..')
sys.path.append('../..')
from utils.utils import (get_dates_dataframe,
                       add_year_week,
                       custom_read,
                       process_columns,
                       remove_accents)

from utils.drive_toolbox import(from_drive_to_local,
                             get_last_modification_date_drive,
                             create_sheets_in_drive_folder,
                             update_sheets_in_drive_folder,
                             read_from_google_sheets,
                             list_file_ids_for_drive_folder,
                             create_csv_file_in_drive_folder,
                             write_csv_to_drive,
                             read_csv_from_drive,
                             append_dataframe_to_google_sheet_from_range,
                             send_google_chat_notification)
from utils.src.constants import (atlas_consumo_output_folder_id,
                           consumo_sheets_ids_dict,
                           folder_id_bauto_gabo,
                           id_reporte_ventas,
                           id_edas_referenciados,
                           id_torre_de_control
                           )


warnings.filterwarnings('ignore')



In [ ]:
# EXTRAS

import math
from zoneinfo import ZoneInfo
import re

# --------- BUSCAR EN SUBCARPETAS -------------------------------------------
from googleapiclient.discovery import build
creds, _ = default()

servicedrive = build("drive", "v3", credentials=creds)
service_sheets = build("sheets", "v4", credentials=creds)

FOLDER_MIME = "application/vnd.google-apps.folder"

def listar_archivos(folder_id, mime_types=None):
    """
    folder_id: ID de la carpeta raíz
    mime_types: None | string | lista de strings
    """
    if isinstance(mime_types, str):
        mime_types = [mime_types]

    resultados = {}

    def recorrer(fid):
        page_token = None
        while True:
            resp = servicedrive.files().list(
                q=f"'{fid}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()

            for f in resp.get("files", []):
                if f["mimeType"] == FOLDER_MIME:
                    recorrer(f["id"])
                else:
                    if mime_types is None or f["mimeType"] in mime_types:
                        resultados[f["name"]] = f["id"]

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    recorrer(folder_id)
    return resultados


# -------------- LEER CON ENCODING ------------------------------------------
import io
import pandas as pd
from googleapiclient.http import MediaIoBaseDownload

def read_csv_from_drive_v3(drive_service, file_id, **read_csv_kwargs):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()

    fh.seek(0)
    return pd.read_csv(fh, **read_csv_kwargs)


# import io
# def read_csv_from_drive2(drive, file_id, encoding="latin-1", **read_csv_kwargs):
#     f = drive.CreateFile({"id": file_id})
#     f.FetchContent()
#     b = f.content.getvalue()  # bytes
#     return pd.read_csv(io.BytesIO(b), encoding=encoding, **read_csv_kwargs)

# ------------- Todas las columnas ------------------------------------------
pd.set_option('display.max_columns', 100)

# ------------- IMPRIMIR CON COLORES ----------------------------------------
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[94m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

def borrar_hojas(spreadsheet_id, nb_hojas):
    spreadsheet = service_sheets.spreadsheets().get(
        spreadsheetId=spreadsheet_id
    ).execute()

    sheet_ids = []
    for sheet in spreadsheet["sheets"]:
        if sheet["properties"]["title"] in nb_hojas:
            sheet_ids.append(sheet["properties"]["sheetId"])

    request = {
        "requests": [
            {"deleteSheet": {"sheetId": s_id}}
            for s_id in sheet_ids
            ]
    }

    service_sheets.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body=request
    ).execute()
    print(f"{nb_hojas} eliminada(s)")
    return

    print("Hoja no encontrada")

def crear_hojas_sheets(spreadsheet_id, nb_hojas, quitar_cuadricula = True, fila_congelada = 1):
  """
  nb_hojas: lista con los nombres de las hojas a crear
  quitar_cuadricula: True | False es para hacer invisibles los bordes/cuadrícula de las celdas
  """

  request = {
      "requests": [
          {"addSheet": {"properties": {"title": nombre,
                                       'gridProperties': {'hideGridlines':quitar_cuadricula,
                                                          'frozenRowCount':fila_congelada}
                                       }
                        }
           }
          for nombre in nb_hojas
      ]
  }

  service_sheets.spreadsheets().batchUpdate(
      spreadsheetId = spreadsheet_id,
      body=request
  ).execute()
  print(f'{nb_hojas} creadas')


def formato_hojas_sheets(sheets_id, nb_hojas, n_columnas, tamanio_letra = 11, letra = 'Source Serif 4', rgb_encabezado = [0.1, 0.3, 0.7], ):
  spreadsheet = service_sheets.spreadsheets().get(
      spreadsheetId=sheets_id
  ).execute()

  sheet_ids = []
  for sheet in spreadsheet["sheets"]:
      if sheet["properties"]["title"] in nb_hojas:
          sheet_ids.append(sheet["properties"]["sheetId"])

  # requests de formato
  requests = []
  for sh_id in sheet_ids:
    # A) Fuente para TODA la hoja
    r_global = {
        "repeatCell": {
            "range": {
                "sheetId": sh_id
            },
            "cell": {
                "userEnteredFormat": {
                    "textFormat": {
                        "fontFamily": letra,
                        "fontSize": tamanio_letra
                    }
                }
            },
            "fields": "userEnteredFormat.textFormat(fontFamily,fontSize)"
        }
    },

    # B) Color solo en encabezado (fila 1)
    r_encabezado = {
        "repeatCell": {
            "range": {
                "sheetId": sh_id,
                "startRowIndex": 0,
                "endRowIndex": 1,
                'startColumnIndex':0,
                'endColumnIndex':n_columnas
            },
            "cell": {
                "userEnteredFormat": {
                    "backgroundColor": {
                        "red": rgb_encabezado[0],
                        "green": rgb_encabezado[1],
                        "blue": rgb_encabezado[2]
                    },
                    "textFormat": {
                        "bold": True,
                        "foregroundColor": {
                            "red": 1,
                            "green": 1,
                            "blue": 1
                        }
                    }
                }
            },
            "fields": "userEnteredFormat(backgroundColor,textFormat.bold,textFormat.foregroundColor)"
        }
    }
    r_anchoColumnas = {
        "autoResizeDimensions": {
            "dimensions": {
                "sheetId": sh_id,
                "dimension": "COLUMNS",
                "startIndex": 0,
                "endIndex": 20   # ajusta primeras 20 columnas
            }
        }
    }
    requests.append(r_global)
    requests.append(r_encabezado)
    requests.append(r_anchoColumnas)

  service_sheets.spreadsheets().batchUpdate(
      spreadsheetId=sheets_id,
      body={"requests": requests}
      ).execute()

In [ ]:
def reasignar_esp_apagados(df_leads, col_espacio_leads, df_asesores, espacios = ['torre','samara','patriotismo','gran sur']):
  """
  df_leads -> leads nuevos. Debe contener al menos la columna col_espacio_leads
  col_espacio_leads -> columna en df_leads de espacio preasignado por SF o por proceso previo, en la que se sobreescriben las reasignaciones
  df_asesores -> catálogo de asesores activos. Debe contener la columna "espacio".
  espacios -> espacios físicos que se muestran en el catálogo de asesores (con el mismo nombre que aparecen ahí), tanto activos como inactivos
  """
  espacios_apagados = [c for c in espacios if len(df_asesores[df_asesores.espacio==c])==0]
  if len(espacios_apagados)>0:
    espacios_activos = [c for c in espacios if c not in espacios_apagados]
    print(f'Espacio(s) apagado(s): {espacios_apagados}. Se redistribuyen sus pedidos en los otros espacios activos: {espacios_activos}')
    leap = df_leads[col_espacio_leads].isin(espacios_apagados) # Leads en Espacio Apagado
    n = leap.sum()
    df_leads.loc[leap, col_espacio_leads] = np.random.choice(espacios_activos, size = n)
    return df_leads
  else:
    print('Todos los espacios están activos')
    return df_leads

## Leemos base de solicitudes y nos quedamos con los nuevos desde el último corte

In [ ]:
fh_corte = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H:%M')
insumos_edas = list_file_ids_for_drive_folder(drive, id_drive_edas)

# Cargamos la base viva (formulario) de folios edas y damos formato al folio
edas_full = read_from_google_sheets(gc, id_sheets_formulario_edas, 'Respuestas de formulario 1')
edas_full['Folio Preautorizado'] = pd.to_numeric(edas_full['Folio Preautorizado'].astype(str).str.strip(), errors = 'coerce').astype('Int64')
edas_full['fh_corte'] = fh_corte
edas_full['fh_corte'] = pd.to_datetime(edas_full['fh_corte'],format='%d-%m-%Y %H:%M')

# Cargamos el corte anterior y damos formato al folio
edas_ultimoCorte = read_csv_from_drive(drive, insumos_edas[nb_ultimo_corte])
edas_ultimoCorte['Folio Preautorizado'] = pd.to_numeric(edas_ultimoCorte['Folio Preautorizado'].astype(str).str.strip(), errors = 'coerce').astype('Int64')
edas_ultimoCorte['fh_corte'] = pd.to_datetime(edas_ultimoCorte['fh_corte'],format='%d-%m-%Y %H')

In [ ]:
edas_ult = edas_full.copy()
edas_penult = edas_ultimoCorte.copy()

cols_edas = {'Folio Preautorizado':'folio','Nombre de Cliente':'nb_comprador','Espacio':'espacio', 'Observaciones de contacto':'obs_contacto','Teléfono celular del cliente':'phone'
          ,'Fecha nacimiento folio ':'fh_creacion_folio','Medio de contacto preferencia del cliente':'pref_contacto'}
edas_ult.rename(columns = cols_edas, inplace=True)
edas_penult.rename(columns = cols_edas, inplace=True)

# Hacemos la diferencia vs el último corte
sols_edas = edas_ult.copy()[~edas_ult['folio'].isin(edas_penult['folio'].unique())].reset_index(drop=True)
sols_edas['tp_solicitud'] = 'EDA Crédito'

# Sólo registros con folio numérico (pasaron por to_numeric en celda anterior). Esto quita folios no válidos y los valores vacíos hasta abajo (Estrellita suele aplicar una fórmula sobre filas vacías)
sols_edas = sols_edas[sols_edas['folio'].notna()]

print(f'{color.CYAN} Tenemos {sols_edas.shape[0]} solicitudes nuevas. \n Falta validar teléfono y monto mayor a 100k {color.END}')

display(sols_edas)
pdds_sf = sols_edas.copy()

In [ ]:
# CELDA DE VALIDACIÓN
fh_final_sols, fh_inicial_sols = edas_ult['fh_corte'].max(), edas_penult['fh_corte'].max()
fhs_creacion_ls = pd.to_datetime(pdds_sf['Marca temporal'],
                                 format = '%d/%m/%Y %H:%M:%S', errors = 'coerce').unique().tolist()

# if not all([x <= fh_final_sols and x >= fh_inicial_sols for x in fhs_creacion_ls if x!=np.nan]):
extemporaneos = ~pd.to_datetime(pdds_sf['Marca temporal'],format = '%d/%m/%Y %H:%M:%S', errors = 'coerce').between(fh_inicial_sols,fh_final_sols)
if extemporaneos.sum()>0:
    print(f"{color.RED}Hay {extemporaneos.sum()} solicitud(es) con fechas fuera del rango de cortes de actualización. Probablemente agregaron folio a una solicitud anterior:{color.END}")
    display(pdds_sf[extemporaneos])
    # raise SystemExit

## Pegamos datos de clientes

In [ ]:
files_glomo = list_file_ids_for_drive_folder(drive, id_drive_glomo)
bases_glomo = [nb_ctes_glomo_mm,nb_ctes_glomo_ma]
bases = {}
def fix_mojibake(text):
    try:
        return text.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError, AttributeError):
        return text

for base in bases_glomo:
  print(f'Leyendo {base}')
  id = files_glomo[base]
  df = read_csv_from_drive(drive, id)
  try:
    df = df[['Número de Solicitud','Fecha de Ingreso','Monto Solicitado','Teléfono Celular']]
  except KeyError as e:
    try:
      print(f'  Corrigiendo {e} por mojibakes')
      df.columns = df.columns.map(fix_mojibake)
      df = df[['Número de Solicitud','Fecha de Ingreso','Monto Solicitado','Teléfono Celular']]
    except KeyError as e:
      print(f'{color.RED}     Columnas no encontradas en {base}{e}:{color.END}')
      print(df.columns.tolist())
      raise
  bases[base] = df
cts_glomo = pd.concat(bases.values())

cts_glomo.columns = ['folio_glomo','fecha_ingreso_glomo','monto_glomo','tel_glomo']

cts_glomo['folio_glomo'] = pd.to_numeric(cts_glomo['folio_glomo'].astype(str).str.strip(), errors = 'coerce').astype('Int64')
cts_glomo.drop_duplicates(subset='folio_glomo', keep='first', inplace=True)
cts_glomo['tel_glomo']=pd.to_numeric(cts_glomo['tel_glomo'], errors = 'coerce').astype('Int64')

In [ ]:
# Cruzamos los tres campos que interesan a estrellita de una vez
pdds_sf = sols_edas.copy()

# Eliminamos teléfono de formulario y extraemos telefono de observaciones de contacto, cuando es posible
pdds_sf = pdds_sf.drop(columns = ['phone'])
phone_regex = re.compile(r'([0-9\s\+-]{8,20})') # patrón consiste sólo en números, espacios, "+" y guiones. Entre 8 y 20 carácteres.
pdds_sf['phone'] = pdds_sf['obs_contacto'].str.strip().str.replace(r'[()-]','', regex = True # sólo quitamos paréntesis y guiones
                                                                                ).str.extract(phone_regex) # extraemos patrón
pdds_sf['phone'] = pdds_sf['phone'].str.replace(r'\D','', regex=True) # Quitamos todo lo que NO sea número
pdds_sf['phone'] = pd.to_numeric(pdds_sf['phone'], errors = 'coerce').astype('Int64')

# Cruce con base glomo
pdds_sf1 = pdds_sf.merge(cts_glomo, how = 'left', left_on = 'folio', right_on = 'folio_glomo')
pdds_sf1['phone'] = np.where(pdds_sf1['phone'].notna(), pdds_sf1['phone'], pdds_sf1['tel_glomo'])
pdds_sf1 = pdds_sf1.drop(columns = ['tel_glomo','folio_glomo'])

# Formato a campo de teléfono
pdds_sf1['phone'] = pd.to_numeric(pdds_sf1['phone'].replace(r'[-\s]', '', regex=True), errors = 'coerce').astype('Int64')

# Validamos teléfono. Si no tiene no se intenta asignar y se elimina del corte para dejarlo al siguiente ejercicio
sin_fon = pdds_sf1['phone'].isna()
if sin_fon.sum()>0:
  print(f'{color.BOLD}{color.RED}Hay {sin_fon.sum()} folios nuevos sin número de teléfono válido {color.END}')
  display(pdds_sf1[sin_fon].reset_index(drop=True))
  # raise SystemExit # No se detiene el proceso
  folio_out_corte = pdds_sf1.loc[sin_fon]['folio'].unique()  # Se elimina del corte para cacharlo en el siguiente ejercicio
  pdds_sf1 = pdds_sf1.loc[~sin_fon] # No se asigna en este ejercico
else:
  print(f'{color.BOLD}{color.GREEN} Todos los folios nuevos tienen número de teléfono válido {color.END}')

# Validamos monto del crédito mayor a 100k. Si no cumplen no se asignan, pero estos no se eliminan del corte
if validar_montos == 'S':
  pdds_sf1['monto_glomo'] = pd.to_numeric(pdds_sf1['monto_glomo'].replace(',','',regex=True), errors='coerce').astype(float)
  print(f'{color.BOLD}{color.RED} Hay {pdds_sf1[~(pdds_sf1['monto_glomo'] >= 99000)].shape[0]} solicitudes por montos menores a 99k, con base en glomo. {color.END}')
  pdds_sf1 = pdds_sf1[pdds_sf1['monto_glomo'] >= 99000]
  print(f'{color.BOLD}{color.CYAN} Quedan {pdds_sf1.shape[0]} que, de acuerdo al reporte de glomo, son por montos mayores a 100k{color.END}')

## Buscamos datos de solicitudes nuevas en histórico y nos quedamos con las que cumplan definición de leads nuevos

In [ ]:
# Leemos y ordenamos histórico, corregimos nombre de asesora y generamos variable de último lead
id_hist = '1zvW-Dxow9gz1Dnbpg_jO7my4wadDvJDW'
nb_hist = '_latest.csv'
hist = list_file_ids_for_drive_folder(drive, id_hist)
hist = [v for i,v in hist.items() if nb_hist in i ][0]
hist = read_csv_from_drive_v3(servicedrive, hist, encoding='latin-1' )

# Primero los registros más recientes
hist['fecha de asignacion'] = pd.to_datetime(hist['fecha de asignacion'], format = '%Y-%m-%d')
hist = hist.sort_values(by='fecha de asignacion', ascending=False)
#------------------------------------

hist['asesor espacio'] = hist['asesor espacio'].str.replace('SaldaÃ±a','Saldaña')
hist['conteo_leads'] = hist['id lead'].str.replace('|','0').str[-6:].astype(int)
ultimo_lead = hist['conteo_leads'].max()
print(f'{color.BLUE}El último lead, a partir del cual vamos a empezar a asignar en este proceso es el {ultimo_lead}{color.END}')
hist.sample(1)

In [ ]:

hist_fon = hist[['telefono comprador','id lead','estatus de lead']].copy().drop_duplicates('telefono comprador').dropna(subset='telefono comprador')

hist_fon.columns = ['phone','id_lead','estatus_lead']

pdds_sf1['fon'] = pd.to_numeric(pdds_sf1['phone'].astype(str).str.strip().str[-10:], errors = 'coerce').astype('Int64')
hist_fon['phone'] = pd.to_numeric(hist_fon['phone'].astype(str).str.strip().str[-10:], errors = 'coerce').astype('Int64')

pdds_sf2 = pdds_sf1.merge(hist_fon[['phone','id_lead','estatus_lead']], how = 'left', on = 'phone', suffixes = ['','_confon'])

cerrados = ['COMPRA EXITOSA ','COMPRA EXITOSA','CERRADO','NA']

nvos_leads = ( (pdds_sf2['id_lead'].isna()) | (pdds_sf2['estatus_lead'].isin(cerrados) ) )

leads_ok = pdds_sf2[~nvos_leads].copy()
leads_nvos = pdds_sf2[nvos_leads].copy()

print(len(leads_ok), len(leads_nvos), len(pdds_sf2))

if not len(leads_ok) + len(leads_nvos) == len(pdds_sf2):
  print(f"{color.BOLD}{color.CYAN}La clasificación de leads ok (pedidos que no requieren lead nuevo) y leads nuevos está perdiendo alguno(s) de los pedidos con los que iniciamos {color.END}")
  raise SystemExit

In [ ]:
leads_nvos['id_comprador'] = 'x'

email_regex = re.compile(r'([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})')
leads_nvos['email'] = leads_nvos['obs_contacto'].astype(str).str.lower().str.extract(email_regex)
leads_nvos['email'] = np.where(leads_nvos['email'].isna(),'x',leads_nvos['email'])

leads_nvos = leads_nvos[['id_comprador','phone','email','nb_comprador','tp_solicitud','folio']].drop_duplicates(['phone'], keep='first').reset_index(drop=True)

print(f'{color.BOLD}{color.CYAN} Tenemos {len(leads_nvos)} leads nuevos.{color.END}')
leads_nvos

## Asignaciones

In [ ]:
# Leemos catálogo de asesores
cat_as = list_file_ids_for_drive_folder(drive, id_drive_catalogos)['AsesoresEspacio']
centros = ['torre','samara','patriotismo','gran sur','celula_credito']
assrs_actvs = {}
for c in centros:
  df = read_from_google_sheets(gc,cat_as,c)
  df['espacio'] = c
  df = df[df.activo==1].drop_duplicates(['asesor','espacio'])
  assrs_actvs[c] = df
assrs_actvs = pd.concat(assrs_actvs.values())
assrs_actvs = assrs_actvs.reset_index(drop=True)
assrs_actvs['asesor'] = assrs_actvs['asesor'].str.lower().str.strip()

# Número de asesores activos por espacio
num_assrs = {c: len(assrs_actvs[assrs_actvs.espacio==c]) for c in assrs_actvs.espacio.unique()}
num_assrs

In [ ]:
# Leads activos por asesor

leads_asesor_espacio = hist[~hist['estatus de lead'].isin(['COMPRA EXITOSA ','COMPRA EXITOSA', 'CERRADO'])].groupby(['asesor espacio','espacio automarket']).agg({'id lead':'nunique'}).reset_index()
leads_asesor_espacio.columns = ['asesor','espacio','leads']
leads_asesor_espacio['asesor'] = leads_asesor_espacio['asesor'].str.lower().str.strip()
leads_asesor_espacio['espacio'] = leads_asesor_espacio['espacio'].replace(dicc_espacios)
leads_asesor_espacio = leads_asesor_espacio[~leads_asesor_espacio['asesor'].str.strip().isin(['PRUEBA','prueba','#N/A()','#n/a ()'])]
leads_asesor_espacio = leads_asesor_espacio.groupby(['asesor','espacio']).agg({'leads':'sum'}).reset_index()

leads_asesor_cred = hist[~hist['estatus de lead'].isin(['COMPRA EXITOSA ','COMPRA EXITOSA', 'CERRADO'])].groupby(['asesor credito']).agg({'id lead':'nunique'}).reset_index()
leads_asesor_cred.columns = ['asesor','leads']
leads_asesor_cred['asesor'] = leads_asesor_cred['asesor'].str.lower().str.strip()
leads_asesor_cred['espacio'] = 'celula_credito'
leads_asesor_cred = leads_asesor_cred[~leads_asesor_cred['asesor'].str.strip().isin(['PRUEBA', 'prueba', '#N/A()', '#n/a ()', '', ' '])]
leads_asesor_cred = leads_asesor_cred.groupby(['asesor','espacio']).agg({'leads':'sum'}).reset_index()

leads_asesor = pd.concat([leads_asesor_espacio, leads_asesor_cred])

assrs_actvs_leads = assrs_actvs.merge(leads_asesor, how = 'left', on = ['asesor','espacio'])
assrs_actvs_leads = assrs_actvs_leads.sort_values(['espacio','leads']).reset_index(drop=True)

assrs_actvs_leads['llave'] = assrs_actvs_leads.groupby('espacio').cumcount()
assrs_actvs_leads['leads'] = assrs_actvs_leads['leads'].fillna(0)

### Asignación espacio

In [ ]:
# Primero asignamos espacio

hist_credito_activos = hist.copy()
hist_credito_activos = hist_credito_activos[hist_credito_activos['origen automarket'].str.contains('EDA') &
                                             ((~hist_credito_activos['estatus de lead'].isin(cerrados)) & (hist_credito_activos['espacio automarket']!='PRUEBA'))]

leads_credito_activos = hist_credito_activos.groupby(['espacio automarket']).agg({'id lead':'nunique'}).reset_index()
leads_credito_activos = leads_credito_activos.sort_values(by='id lead').reset_index(drop=True)
leads_credito_activos['diff_leads'] = leads_credito_activos['id lead'].diff().shift(-1).fillna(0).astype(int)

# Esta linea es importante, asigna el peso definido a la carga de trabajo relativa
leads_credito_activos['diff_leads'] = round(leads_credito_activos['diff_leads']*nivelar_carga_espacios)
# --------------------------------------------------------------------------------
leads_credito_activos

In [ ]:
# df de asignación para emparejar los leads activos en espacios
asign_espacio_justiciera = leads_credito_activos.loc[
    leads_credito_activos.index.repeat(leads_credito_activos["diff_leads"])
]['espacio automarket'].reset_index(drop=True) # sale como una serie
asign_espacio_justiciera = asign_espacio_justiciera.to_frame()

# df de asignación normal
asign_espacio_normal = assrs_actvs[assrs_actvs['espacio']!='celula_credito'].drop_duplicates('espacio').rename(columns = {'espacio':'espacio automarket'}
                                                          )['espacio automarket'].reset_index(drop=True).to_frame()

# Ahora vamos a pegar tantas veces como sea necesario la asignación normal
lte = len(leads_nvos) - len(asign_espacio_justiciera) # leads tras emparejamiento
repeticiones_carrusel = ( math.ceil(lte/len(asign_espacio_normal)) ) if lte>0 else 0
print(f'Después de emparejar los leads de crédito en cada espacio, vamos a pasarlos {repeticiones_carrusel} veces por el carrusel')

asign_espacio = asign_espacio_justiciera.copy()
for i in range(repeticiones_carrusel):
  asign_espacio = pd.concat([asign_espacio,asign_espacio_normal])
asign_espacio.reset_index(drop=True, inplace = True)
asign_espacio['llave_espacio'] = asign_espacio.index + 1
asign_espacio

In [ ]:
leads_nvos = leads_nvos.reset_index(drop=True)
leads_nvos['llave_espacio'] = leads_nvos.index + 1
leads_nvos = leads_nvos.merge(asign_espacio, how='left', on = 'llave_espacio')
leads_nvos['espacio automarket'] = leads_nvos['espacio automarket'].replace(dicc_espacios)

leads_nvos = leads_nvos.rename(columns = {'espacio automarket':'espacio asig'}) #espacio asig es el espacio que asignamos en el paso previo

In [ ]:
# Vamos a procesar distinto los leads que ya tuvieron gestión en algún espacio. Los seleccionamos con inner merge vs histórico

leads_nvos['phone'] = pd.to_numeric(leads_nvos['phone'],errors = 'coerce').astype('Int64')
hist['telefono comprador'] = pd.to_numeric(hist['telefono comprador'],errors = 'coerce').astype('Int64')
leads_nvos_comprprevio = leads_nvos.copy()
leads_nvos_comprprevio = leads_nvos_comprprevio.merge(
    hist[['telefono comprador','espacio automarket','asesor espacio']].rename(
    columns = {'espacio automarket':'espacio previo', 'asesor espacio':'asesor espacio previo'}
    ).drop_duplicates(
        'telefono comprador',keep='first'),
                                        how = 'inner', left_on = 'phone', right_on = 'telefono comprador').drop(columns = ['telefono comprador'])

leads_nvos_comprprevio['espacio previo'] = leads_nvos_comprprevio['espacio previo'].replace(dicc_espacios)

# Si hay espacios apagados, reasignamos sus leads
leads_nvos_comprprevio = reasignar_esp_apagados(df_leads=leads_nvos_comprprevio, col_espacio_leads='espacio previo', df_asesores=assrs_actvs)

In [ ]:
# Los leads de compradores nuevos sólo reasignan si hay espacios apagados

leads_nvos_comprnvo = leads_nvos[~leads_nvos['phone'].isin(leads_nvos_comprprevio['phone'].unique())].copy()
leads_nvos_comprnvo = reasignar_esp_apagados(df_leads=leads_nvos_comprnvo, col_espacio_leads='espacio asig', df_asesores=assrs_actvs)
leads_nvos_comprnvo['llave_esp'] = leads_nvos_comprnvo.groupby('espacio asig').cumcount() % leads_nvos_comprnvo['espacio asig'].map(num_assrs)

### Asesores Espacio

In [ ]:
leads_asesor_esp = assrs_actvs_leads[assrs_actvs_leads['espacio'] != 'celula_credito'].rename(columns = {'asesor':'asesor espacio'})
display(leads_asesor_esp)

In [ ]:
# Leads con comprador previo se quedan, del espacio previo, con cualquier asesor distinto al que los gestionó (primer asesor que cruce y no haya tenido antes [columna aux])

leads_nvos_comprprevio = leads_nvos_comprprevio.merge(leads_asesor_esp.rename(columns = {'asesor espacio':'asesor espacio nvo'}), how = 'left', left_on = 'espacio previo', right_on = 'espacio')
leads_nvos_comprprevio['aux'] = (leads_nvos_comprprevio['asesor espacio previo'].str.lower().str.strip() == leads_nvos_comprprevio['asesor espacio nvo'].str.lower().str.strip())*1
leads_nvos_comprprevio = leads_nvos_comprprevio.sort_values(by='aux', ascending=True)
leads_nvos_comprprevio = leads_nvos_comprprevio.drop_duplicates(['phone'], keep = 'first').reset_index(drop=True)
leads_nvos_comprprevio = leads_nvos_comprprevio.drop(columns = ['espacio asig','espacio previo','asesor espacio previo','activo','leads','llave','aux']).rename(columns = {'asesor espacio nvo':'asesor espacio'})
display(leads_nvos_comprprevio)

# Leads con comprador nuevo se asignan de acuerdo al orden del número de leads por asesor, dentro de cada espacio (columna llave de leads_asesor_esp)

leads_nvos_comprnvo = leads_nvos_comprnvo.merge(leads_asesor_esp, how='left', left_on = ['espacio asig','llave_esp'], right_on = ['espacio','llave'])
display(leads_nvos_comprnvo)

# Juntamos nuevamente todos los leads nuevos en un df
salida_leads = pd.concat([leads_nvos_comprnvo, leads_nvos_comprprevio]).sort_values(by='id_comprador').reset_index(drop=True)

### Asesores Crédito

In [ ]:
leads_asesor_cred = assrs_actvs_leads[assrs_actvs_leads['espacio'] == 'celula_credito'].rename(columns = {'asesor':'asesor credito'})
display(leads_asesor_cred)

In [ ]:
# Asignamos asesores de crédito de acuerdo a [llave] en leads_asesor_cred, que se define por el orden del número de leads activos del asesor

salida_leads['llave_celcred'] = salida_leads.index % num_assrs['celula_credito']
salida_leads = salida_leads.merge(leads_asesor_cred, how = 'left', left_on = 'llave_celcred', right_on = 'llave', suffixes = ['','_cred'])
salida_leads

## Formato de salidas

In [ ]:
# Leemos catálogo de nomenclatura de leads
cat_nomLeads = list_file_ids_for_drive_folder(drive, id_drive_catalogos)['NomenclaturaLeads']
cat_nomLeads = read_from_google_sheets(gc, cat_nomLeads)
cat_nomLeads = cat_nomLeads[['Tipo de Lead','Clave']]

In [ ]:
salida_leads = salida_leads.reset_index(drop=True)
salida_leads['index'] = salida_leads.index + 1
salida_leads['id lead'] = ultimo_lead + salida_leads['index']
salida_leads = salida_leads.merge(cat_nomLeads, how = 'left', left_on = 'tp_solicitud', right_on = 'Tipo de Lead')
# Validamos que todos tengan nomenclatura asignada
if (salida_leads['Clave'].isna().sum() > 0):
  print(f'{color.RED} Algo falló en la nomenclatura de leads. Hay algunos sin clave{color.END}')
  raise SystemExit

salida_leads['id lead'] = salida_leads['Clave'] + '-' + salida_leads['id lead'].astype(str).str.zfill(6)
salida_leads['folio'] = salida_leads['folio'].fillna('x')
salida_leads.drop(columns = ['llave','index','activo','leads'], inplace = True)
salida_leads['cosecha'] = cosecha
salida_leads['fecha de asignacion'] = fh_de_asignacion.replace('-','/')

salida_leads.rename(columns = {'id_comprador':'id comprador','espacio':'espacio automarket','phone':'telefono comprador','email':'mail comprador',
                               'nb_comprador':'nombre comprador','tp_solicitud':'origen automarket','folio':'folio bauto tc'},inplace=True)
salida_leads = salida_leads[['id lead','origen automarket','cosecha','id comprador','folio bauto tc',
                             'nombre comprador','mail comprador','telefono comprador','asesor credito','espacio automarket','asesor espacio','fecha de asignacion']]

salida_leads['espacio automarket'] = salida_leads['espacio automarket'].replace(dicc_espacios3)
salida_leads['asesor credito'] = salida_leads['asesor credito'].str.title()
salida_leads['asesor espacio'] = salida_leads['asesor espacio'].str.title()
salida_leads['estatus de lead'] = 'celula de credito'

salida_leads

In [ ]:
# Leads activos por asesor iniciales
hca = hist_credito_activos[['espacio automarket','asesor espacio','asesor credito','origen automarket','id lead']]
hca['espacio automarket'] = hca['espacio automarket'].replace(dicc_espacios)
hca['origen automarket'] = hca['origen automarket'].str.strip().str.split(' ').str[0] # Nos quedamos con la primera palabra
hca = hca[~hca['asesor espacio'].str.strip().isin(['PRUEBA','prueba','#N/A()','#n/a ()'])]

hca_ases = hca.copy().rename(columns = {'id lead':'leads','asesor espacio':'asesor','espacio automarket':'espacio'})
hca_ases = hca_ases.pivot_table(index=['espacio', 'asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
hca_ases.columns = hca_ases.columns.map(lambda col: '_'.join([str(x) for x in col if x]))

hca_ascr = hca.copy().rename(columns = {'id lead':'leads','asesor credito':'asesor'})
hca_ascr = hca_ascr.pivot_table(index=['asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
hca_ascr.columns = hca_ascr.columns.map(lambda col: '_'.join([str(x) for x in col if x]))
hca_ascr['espacio'] = 'celula_credito'

hca = pd.concat([hca_ases, hca_ascr])
hca['asesor'] = hca['asesor'].str.lower().str.strip()
assrs_actvs_leads = assrs_actvs.merge(hca, how = 'left', on = ['asesor','espacio'])
assrs_actvs_leads = assrs_actvs_leads[['espacio','asesor','activo'] + [c for c in assrs_actvs_leads.columns if c not in ['espacio','asesor','activo']]].fillna(0)
assrs_actvs_leads['espacio'] = assrs_actvs_leads['espacio'].replace(dicc_espacios3)

# Leads asignados por asesor

res_asign = salida_leads.copy()
res_asign['asesor espacio'] = res_asign['asesor espacio'].str.lower().str.strip()
res_asign['asesor credito'] = res_asign['asesor credito'].str.lower().str.strip()
res_asign['origen automarket'] = res_asign['origen automarket'].str.strip().str.split(' ').str[0]

asgns_ases = res_asign.copy().rename(columns = {'espacio automarket':'espacio','asesor espacio':'asesor','id lead':'leads'})
asgns_ases = asgns_ases.pivot_table(index=['espacio','asesor'],columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
asgns_ases.columns = asgns_ases.columns.map(lambda col: '_'.join([str(x) for x in col if x]))

asgns_ascr = res_asign.copy().rename(columns = {'espacio automarket':'espacio','asesor credito':'asesor','id lead':'leads'})
asgns_ascr = asgns_ascr.pivot_table(index=['asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
asgns_ascr.columns = asgns_ascr.columns.map(lambda col: '_'.join([str(x) for x in col if x]))
asgns_ascr['espacio'] = 'celula_credito'

res_asign = pd.concat([asgns_ases, asgns_ascr]).fillna(0)

res_asign = assrs_actvs_leads.merge(res_asign, how = 'left', on = ['espacio','asesor'], suffixes = ['_iniciales','_nuevos'])
res_asign.fillna(0,inplace = True)
for c in assrs_actvs_leads.columns:
  if c not in ['espacio','asesor','activo']:
    try:
      res_asign[f'{c}_finales'] = res_asign[f'{c}_iniciales'] + res_asign[f'{c}_nuevos']
    except:
      print(f'No hay asignación nueva de {c}')

In [ ]:
cols_agg = [c for c in res_asign.columns if all(x != c for x in ['espacio','asesor','activo']) ]
sums = res_asign.groupby('espacio').agg({c: 'sum' for c in cols_agg}).reset_index()
stds = res_asign.groupby('espacio').agg({c: 'std' for c in cols_agg}).reset_index()
sums['asesor'] = 'sum'
stds['asesor'] = 'std'
sums['activo'] = 0
stds['activo'] = 0

res_asign = pd.concat([res_asign,sums,stds]).reset_index(drop=True)
res_asign[cols_agg] = res_asign[cols_agg].round(1)
res_asign

In [ ]:
# Celda de validacion tipo assert
id_l_nvos = salida_leads['id lead'].unique()

if not hist[hist['id lead'].isin(id_l_nvos)].shape[0] == 0:
    print(f"{color.RED}Se están duplicando id leads respecto a los que ya existían en la torre de control{color.END}")
    raise SystemExit

## Guardamos reportes de Asignación

In [ ]:
# Creamos la hoja de sheets de cero

ahora_dt = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H:%M')
nb_sheets_salida = f'Salida {ahora_dt}'
nb_hojas = ['asignacion_edas','resumen_asignaciones']

id_folder_mes_salida = list_file_ids_for_drive_folder(drive, id_drive_salidas)[nb_carpeta_ctes_mes]
create_sheets_in_drive_folder(gc, nb_sheets_salida, id_folder_mes_salida)
id_sheets_salida = listar_archivos(id_folder_mes_salida, mime_types='application/vnd.google-apps.spreadsheet')[nb_sheets_salida]

# Creamos hojas y eliminamos la hoja 1.
# Si esto da error, muy probablemente es porque ya hay un archivo con el mismo nombre en la carpeta. -----> Revisa la carpeta de drive.
crear_hojas_sheets(id_sheets_salida, nb_hojas)
borrar_hojas(id_sheets_salida,['Hoja 1'])

In [ ]:
# Guardamos la salida del proceso de asignación en hojas de respaldo

hoja_df = {'asignacion_edas': salida_leads, 'resumen_asignaciones':res_asign}
for hoja, df in hoja_df.items():
  update_sheets_in_drive_folder(gc, id_sheets_salida, hoja, df)
  formato_hojas_sheets(id_sheets_salida, [hoja], n_columnas = df.shape[1], letra = 'Source Serif 4' )

print(f'Las salidas se generaron y se guardaron en https://docs.google.com/spreadsheets/d/{id_sheets_salida}')

## Actualizar Torre de Control y corte de EDAs

In [ ]:
# Guardamos el corte con el que trabajamos en esta asignación

# primero el df de los edas con corte de ejercicio previo, después el full con corte de este ejercicio
corte = pd.concat([edas_ultimoCorte, edas_full])
# fh_corte cambia en los cortes, por lo que la excluimos del criterio de duplicidad
corte['Folio Preautorizado'] = pd.to_numeric(corte['Folio Preautorizado'].astype(str).str.strip(),errors='coerce').astype('Int64')
corte = corte.drop_duplicates('Folio Preautorizado',
                              # priorizamos los que ya estaban en el último corte (por su etiqueta de fh_corte)
                              keep='first')
if sin_fon.sum()>0:
  corte = corte.loc[~corte['Folio Preautorizado'].isin(folio_out_corte)]

# Contra este se sacará la diferencia de folios en la siguiente asignación
write_csv_to_drive(drive, insumos_edas[nb_ultimo_corte], corte)

#Este contiene lo mismo, pero sólo es para respaldo
create_csv_file_in_drive_folder(drive, id_drive_edas, corte, f'corteEdas_{ahora_dt}.csv')

In [ ]:
# Tomamos foto a torre de control
ahora_dt = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H:%M')
nb_foto = f'FotoTC_{ahora_dt}'
tc2_foto = read_from_google_sheets(gc, id_sheets_tc2, sheetname='asignacion')

# guardamos respaldo
crear_hojas_sheets(id_sheets_salida, [nb_foto])
update_sheets_in_drive_folder(gc, id_sheets_salida, nb_foto, tc2_foto)
formato_hojas_sheets(id_sheets_salida, [nb_foto], n_columnas = tc2_foto.shape[1], letra = 'Source Serif 4' )
# actualizamos si aplica
if actualizar_tc == 'S':
  append_dataframe_to_google_sheet_from_range(gc, id_sheets_tc2, 'asignacion', salida_leads)
else:
  print('No se actualizó la torre de control')

In [ ]:
# # Guardamos el primer corte de edas.
# edas_ultimoCorte = edas_full.copy().reset_index(drop=True)
# edas_ultimoCorte['fh_corte'] = '24-02-2026 18'
# edas_ultimoCorte = edas_ultimoCorte[:-14]
# edas_ultimoCorte
# create_csv_file_in_drive_folder(drive, id_drive_edas, edas_ultimoCorte, nb_ultimo_corte)